In [1]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])

try:
    import yaml
except ImportError as exc:
    raise RuntimeError(
        "PyYAML is required. Activate the course environment or ask your instructor; "
        "this notebook will not install packages."
    ) from exc

print("PyYAML:", yaml.__version__)
print("Workbench ready. Continue to Step 1.")

Python: 3.14.4
PyYAML: 6.0.3
Workbench ready. Continue to Step 1.


In [2]:
WORK = Path("work")
WORK.mkdir(exist_ok=True)

deployment_yaml = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: mp202-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: mp202-api
  template:
    metadata:
      labels:
        app: mp202-api
    spec:
      containers:
        - name: api
          image: mp202-api:1.0
          ports:
            - containerPort: 8000
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 10
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 30
            periodSeconds: 20
            failureThreshold: 3
"""

(WORK / "deployment.yaml").write_text(deployment_yaml, encoding="utf-8")
print(deployment_yaml)

apiVersion: apps/v1
kind: Deployment
metadata:
  name: mp202-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: mp202-api
  template:
    metadata:
      labels:
        app: mp202-api
    spec:
      containers:
        - name: api
          image: mp202-api:1.0
          ports:
            - containerPort: 8000
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 10
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 30
            periodSeconds: 20
            failureThreshold: 3



In [3]:
service_yaml = """apiVersion: v1
kind: Service
metadata:
  name: mp202-svc
spec:
  type: ClusterIP
  selector:
    app: mp202-api
  ports:
    - port: 80
      targetPort: 8000
"""

(WORK / "service.yaml").write_text(service_yaml, encoding="utf-8")
print(service_yaml)

apiVersion: v1
kind: Service
metadata:
  name: mp202-svc
spec:
  type: ClusterIP
  selector:
    app: mp202-api
  ports:
    - port: 80
      targetPort: 8000



In [4]:
with (WORK / "deployment.yaml").open(encoding="utf-8") as file:
    deployment = yaml.safe_load(file)

with (WORK / "service.yaml").open(encoding="utf-8") as file:
    service = yaml.safe_load(file)

selector = deployment["spec"]["selector"]["matchLabels"]
pod_labels = deployment["spec"]["template"]["metadata"]["labels"]
service_selector = service["spec"]["selector"]
container = deployment["spec"]["template"]["spec"]["containers"][0]

print("Deployment:", deployment["metadata"]["name"])
print("Replicas:", deployment["spec"]["replicas"])
print("Image:", container["image"])
print("Service:", service["metadata"]["name"])
print("Deployment selector matches Pod labels:", selector == pod_labels)
print("Service selector matches Pod labels:", service_selector == pod_labels)
print("Service port mapping:", service["spec"]["ports"][0])
print("Readiness delay:", container["readinessProbe"]["initialDelaySeconds"])
print("Liveness delay:", container["livenessProbe"]["initialDelaySeconds"])

Deployment: mp202-api
Replicas: 2
Image: mp202-api:1.0
Service: mp202-svc
Deployment selector matches Pod labels: True
Service selector matches Pod labels: True
Service port mapping: {'port': 80, 'targetPort': 8000}
Readiness delay: 5
Liveness delay: 30


In [5]:
service_v2_yaml = """apiVersion: v1
kind: Service
metadata:
  name: mp202-api-v2-svc
spec:
  type: NodePort
  selector:
    app: mp202-api-v2
  ports:
    - port: 80
      targetPort: 8000
"""

(WORK / "service-v2.yaml").write_text(service_v2_yaml, encoding="utf-8")
with (WORK / "service-v2.yaml").open(encoding="utf-8") as file:
    T1_service = yaml.safe_load(file)

print(T1_service)

{'apiVersion': 'v1', 'kind': 'Service', 'metadata': {'name': 'mp202-api-v2-svc'}, 'spec': {'type': 'NodePort', 'selector': {'app': 'mp202-api-v2'}, 'ports': [{'port': 80, 'targetPort': 8000}]}}


In [6]:
T2_probes = {
    "readinessProbe": {
        "httpGet": {"path": "/health", "port": 8000},
        "initialDelaySeconds": 10,
        "periodSeconds": 10,
    },
    "livenessProbe": {
        "httpGet": {"path": "/health", "port": 8000},
        "initialDelaySeconds": 60,
        "periodSeconds": 20,
        "failureThreshold": 3,
    },
}

print(T2_probes)

{'readinessProbe': {'httpGet': {'path': '/health', 'port': 8000}, 'initialDelaySeconds': 10, 'periodSeconds': 10}, 'livenessProbe': {'httpGet': {'path': '/health', 'port': 8000}, 'initialDelaySeconds': 60, 'periodSeconds': 20, 'failureThreshold': 3}}


In [7]:
broken_deploy = yaml.safe_load("""apiVersion: apps/v1
kind: Deployment
metadata:
  name: payments-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: payments-api
  template:
    metadata:
      labels:
        app: payments-api
    spec:
      containers:
        - name: api
          image: payments-api:3.1
""")

broken_svc = yaml.safe_load("""apiVersion: v1
kind: Service
metadata:
  name: payments-svc
spec:
  selector:
    app: payment-api
  ports:
    - port: 80
      targetPort: 8080
""")

print("Deployment Pod labels:", broken_deploy["spec"]["template"]["metadata"]["labels"])
print("Original Service selector:", broken_svc["spec"]["selector"])

broken_svc["spec"]["selector"] = broken_deploy["spec"]["template"]["metadata"]["labels"]
T3_fixed_service = broken_svc

print("Corrected Service:", T3_fixed_service)

Deployment Pod labels: {'app': 'payments-api'}
Original Service selector: {'app': 'payment-api'}
Corrected Service: {'apiVersion': 'v1', 'kind': 'Service', 'metadata': {'name': 'payments-svc'}, 'spec': {'selector': {'app': 'payments-api'}, 'ports': [{'port': 80, 'targetPort': 8080}]}}


In [8]:
checks = [
    ("Deployment name is mp202-api", deployment["metadata"]["name"] == "mp202-api"),
    ("Deployment has 2 replicas", deployment["spec"]["replicas"] == 2),
    ("Image and container port are correct", container["image"] == "mp202-api:1.0" and container["ports"][0]["containerPort"] == 8000),
    ("Deployment selector matches Pod labels", selector == pod_labels),
    ("Service name is mp202-svc", service["metadata"]["name"] == "mp202-svc"),
    ("Service selector matches Pod labels", service_selector == pod_labels),
    ("Service forwards port 80 to 8000", service["spec"]["ports"][0]["port"] == 80 and service["spec"]["ports"][0]["targetPort"] == 8000),
    ("Both probes call /health on port 8000", all(container[key]["httpGet"] == {"path": "/health", "port": 8000} for key in ("readinessProbe", "livenessProbe"))),
    ("Liveness waits longer than readiness", container["livenessProbe"]["initialDelaySeconds"] > container["readinessProbe"]["initialDelaySeconds"]),
    ("T1 Service name, type, and selector are correct", T1_service["metadata"]["name"] == "mp202-api-v2-svc" and T1_service["spec"]["type"] == "NodePort" and T1_service["spec"]["selector"] == {"app": "mp202-api-v2"}),
    ("T1 forwards port 80 to 8000", T1_service["spec"]["ports"][0]["port"] == 80 and T1_service["spec"]["ports"][0]["targetPort"] == 8000),
    ("T2 probes use the requested delays and endpoint", T2_probes["readinessProbe"]["initialDelaySeconds"] == 10 and T2_probes["livenessProbe"]["initialDelaySeconds"] == 60 and all(probe["httpGet"] == {"path": "/health", "port": 8000} for probe in T2_probes.values())),
    ("T3 Service selector matches Pod labels", T3_fixed_service["spec"]["selector"] == broken_deploy["spec"]["template"]["metadata"]["labels"]),
    ("T3 preserved Service name and target port", T3_fixed_service["metadata"]["name"] == "payments-svc" and T3_fixed_service["spec"]["ports"][0]["targetPort"] == 8080),
]

for label, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'} | {label}")

passed_count = sum(passed for _, passed in checks)
print(f"\n{passed_count} of {len(checks)} checks passed")

PASS | Deployment name is mp202-api
PASS | Deployment has 2 replicas
PASS | Image and container port are correct
PASS | Deployment selector matches Pod labels
PASS | Service name is mp202-svc
PASS | Service selector matches Pod labels
PASS | Service forwards port 80 to 8000
PASS | Both probes call /health on port 8000
PASS | Liveness waits longer than readiness
PASS | T1 Service name, type, and selector are correct
PASS | T1 forwards port 80 to 8000
PASS | T2 probes use the requested delays and endpoint
PASS | T3 Service selector matches Pod labels
PASS | T3 preserved Service name and target port

14 of 14 checks passed
